# Fine-tune Qwen3-1.7B sinh truyện ngụ ngôn tiếng Việt (QLoRA / Unsloth)

Notebook chạy trên **Google Colab (GPU T4 free)**.

**Luồng:** cài đặt → chỉnh **HYPERPARAMETERS** → tải `train.jsonl`/`val.jsonl` (từ `data/processed/`, đã lọc < ~6000 ký tự) → nạp model 4-bit → gắn LoRA → train → sinh thử → merge & tải về.

Sau khi tải `*-merged.zip` về máy: chạy `scripts/export_gguf.sh` rồi `ollama create fable-tuned -f ollama/Modelfile` (Task 8) để dùng trong web app.

In [ ]:
# Cài đặt (Colab T4)
%pip install -q unsloth "trl>=0.9" "transformers>=4.44" "datasets>=2.20"

In [ ]:
# ===== HYPERPARAMETERS — chỉnh tự do trước khi train =====
MODEL_NAME      = "unsloth/Qwen3-1.7B"   # base Qwen3-1.7B (khớp base model của web app)
MAX_SEQ_LENGTH  = 2048                    # khớp lọc dữ liệu < ~6000 ký tự; tăng nếu giữ truyện dài hơn
LOAD_IN_4BIT    = True

# LoRA
LORA_R          = 16
LORA_ALPHA      = 16
LORA_DROPOUT    = 0.0

# Training
LEARNING_RATE   = 2e-4
EPOCHS          = 3
BATCH_SIZE      = 2
GRAD_ACCUM      = 4
WARMUP_STEPS    = 5
SEED            = 42

# Dữ liệu (đã lọc bằng: python scripts/prepare_data.py --max-chars 6000)
TRAIN_PATH      = "train.jsonl"
VAL_PATH        = "val.jsonl"

# Format dữ liệu — khớp với app
SYSTEM_PROMPT   = "Bạn là người kể truyện ngụ ngôn cho trẻ em."
ENABLE_THINKING = False   # tắt thinking để model học sinh truyện trực tiếp (khớp inference think=false)

OUTPUT_DIR      = "outputs"
MERGED_DIR      = "qwen3-1.7b-fable-merged"

In [ ]:
# Tải lên dữ liệu đã xử lý (chọn train.jsonl và val.jsonl từ data/processed/)
from google.colab import files
print("Chọn train.jsonl và val.jsonl:")
uploaded = files.upload()

In [ ]:
# Nạp model 4-bit + gắn LoRA
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

In [ ]:
# Nạp dữ liệu + áp chat template
from datasets import load_dataset

ds = load_dataset("json", data_files={"train": TRAIN_PATH, "val": VAL_PATH})

def to_text(ex):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": ex["instruction"]},
        {"role": "assistant", "content": ex["output"]},
    ]
    try:
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, enable_thinking=ENABLE_THINKING)
    except TypeError:
        # tokenizer không hỗ trợ enable_thinking -> bỏ tham số
        text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

ds = ds.map(to_text)
print(ds)
print(ds["train"][0]["text"][:500])

In [ ]:
# Train (QLoRA / SFT)
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds["train"],
    eval_dataset=ds["val"],
    args=SFTConfig(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=WARMUP_STEPS,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        logging_steps=5,
        eval_strategy="epoch",
        output_dir=OUTPUT_DIR,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        seed=SEED,
    ),
)
trainer.train()

In [ ]:
# Sinh thử (kiểm chứng định tính) — so sánh với base trước khi train nếu muốn
FastLanguageModel.for_inference(model)

msgs = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Viết một truyện ngụ ngôn cho trẻ em về chủ đề: lòng kiên nhẫn. "
                                 "Bài học đạo đức: kiên nhẫn sẽ thành công. Độ tuổi phù hợp: 6-8 tuổi."},
]
try:
    prompt = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True, enable_thinking=ENABLE_THINKING)
except TypeError:
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
out = model.generate(**inputs, max_new_tokens=400, temperature=0.8, top_p=0.9)
print(tokenizer.decode(out[0], skip_special_tokens=True))

In [ ]:
# Merge LoRA vào base + lưu (16-bit) + nén & tải về
model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")

import shutil
shutil.make_archive(MERGED_DIR, "zip", MERGED_DIR)

from google.colab import files
files.download(f"{MERGED_DIR}.zip")

## Bước tiếp theo (trên máy local — Task 8)

1. Giải nén `qwen3-1.7b-fable-merged.zip` vào `models/`.
2. Convert sang GGUF: `./scripts/export_gguf.sh models/qwen3-1.7b-fable-merged models/fable-tuned-q8_0.gguf`
3. Tạo model Ollama: `ollama create fable-tuned -f ollama/Modelfile`
4. Chạy app với `FABLE_BASE_MODEL=qwen3:1.7b FABLE_TUNED_MODEL=fable-tuned` để so sánh before/after.